# GitMind Commit Model — Dataset Exploration

Exploration du dataset CommitChronicle avant la préparation du corpus
d'entraînement de notre modèle de génération de messages de commit.

**Objectif :** analyser un échantillon des trois partitions sans télécharger
le dataset complet.

Dataset : [JetBrains-Research/commit-chronicle](https://huggingface.co/datasets/JetBrains-Research/commit-chronicle)


In [1]:
%pip -q install "datasets>=4,<5" pyyaml


In [2]:
from pathlib import Path
import subprocess

REPO_DIR = Path("/content/gitmind-commit-model")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone",
         "https://github.com/EgueRichnel2025/gitmind-commit-model.git",
         str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        check=True,
    )

%cd /content/gitmind-commit-model


/content/gitmind-commit-model


In [3]:
from collections import Counter
from itertools import islice
from pathlib import Path
import json
import re

import yaml
from datasets import load_dataset

with open("configs/data.yaml", encoding="utf-8") as file:
    CONFIG = yaml.safe_load(file)

DATASET_ID = CONFIG["primary"]["repository"]
DATASET_CONFIG = CONFIG["primary"]["configuration"]
SAMPLE_SIZE = CONFIG["exploration"]["sample_size_per_split"]
SEED = CONFIG["exploration"]["random_seed"]

print("Dataset :", DATASET_ID)
print("Configuration :", DATASET_CONFIG)
print("Taille de chaque échantillon :", SAMPLE_SIZE)


Dataset : JetBrains-Research/commit-chronicle
Configuration : default
Taille de chaque échantillon : 1000


In [4]:
def load_sample(split_name):
    print(f"Chargement de la partition : {split_name}")

    stream = load_dataset(
        DATASET_ID,
        DATASET_CONFIG,
        split=split_name,
        streaming=True,
    )

    print("Structure :", stream.features)

    stream = stream.shuffle(
        seed=SEED,
        buffer_size=10_000,
    )

    records = list(islice(iter(stream), SAMPLE_SIZE))

    print(f"Exemples récupérés : {len(records)}")
    return records


SAMPLES = {}

for split_name in ("train", "validation", "test"):
    SAMPLES[split_name] = load_sample(split_name)


Chargement de la partition : train


README.md:   0%|          | 0.00/8.10k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/61 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/61 [00:00<?, ?it/s]

Structure : {'author': Value('int64'), 'date': Value('string'), 'timezone': Value('int64'), 'hash': Value('string'), 'message': Value('string'), 'mods': List({'change_type': Value('string'), 'old_path': Value('string'), 'new_path': Value('string'), 'diff': Value('string')}), 'language': Value('string'), 'license': Value('string'), 'repo': Value('string'), 'original_message': Value('string')}
Exemples récupérés : 1000
Chargement de la partition : validation


Resolving data files:   0%|          | 0/61 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/61 [00:00<?, ?it/s]

Structure : {'author': Value('int64'), 'date': Value('string'), 'timezone': Value('int64'), 'hash': Value('string'), 'message': Value('string'), 'mods': List({'change_type': Value('string'), 'old_path': Value('string'), 'new_path': Value('string'), 'diff': Value('string')}), 'language': Value('string'), 'license': Value('string'), 'repo': Value('string'), 'original_message': Value('string')}
Exemples récupérés : 1000
Chargement de la partition : test


Resolving data files:   0%|          | 0/61 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/61 [00:00<?, ?it/s]

Structure : {'author': Value('int64'), 'date': Value('string'), 'timezone': Value('int64'), 'hash': Value('string'), 'message': Value('string'), 'mods': List({'change_type': Value('string'), 'old_path': Value('string'), 'new_path': Value('string'), 'diff': Value('string')}), 'language': Value('string'), 'license': Value('string'), 'repo': Value('string'), 'original_message': Value('string')}
Exemples récupérés : 1000


In [6]:
SENSITIVE_PATTERNS = {
    "email": re.compile(
        r"\\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\\.[A-Z]{2,}\\b",
        re.IGNORECASE,
    ),
    "ipv4": re.compile(
        r"\\b(?:[0-9]{1,3}\\.){3}[0-9]{1,3}\\b"
    ),
    "github_token": re.compile(
        r"\\bgh[pousr]_[A-Za-z0-9]{20,}\\b"
    ),
    "aws_key": re.compile(r"\\bAKIA[0-9A-Z]{16}\\b"),
    "private_key": re.compile(
        r"-----BEGIN (?:RSA |EC |OPENSSH )?PRIVATE KEY-----"
    ),
}


def distribution(values):
    values = sorted(values)

    if not values:
        return {}

    def percentile(fraction):
        index = round((len(values) - 1) * fraction)
        return values[index]

    return {
        "count": len(values),
        "min": values[0],
        "median": percentile(0.50),
        "p90": percentile(0.90),
        "p95": percentile(0.95),
        "max": values[-1],
    }


def analyze_split(records):
    languages = Counter()
    licenses = Counter()
    seen_commits = set()

    diff_lengths = []
    message_lengths = []
    modification_counts = []

    empty_diffs = 0
    empty_messages = 0
    duplicate_commits = 0
    same_as_original = 0
    sensitive_counts = Counter()

    previews = []

    for row in records:
        message = str(row.get("message") or "").strip()
        original = str(row.get("original_message") or "").strip()
        modifications = row.get("mods") or []

        diffs = [
            str(mod.get("diff") or "")
            for mod in modifications
            if isinstance(mod, dict)
        ]

        diff_text = "\\n".join(diffs).strip()

        language = str(row.get("language") or "<missing>")
        license_name = str(row.get("license") or "<missing>")
        repo = str(row.get("repo") or "<missing>")
        commit_hash = str(row.get("hash") or "")

        languages[language] += 1
        licenses[license_name] += 1

        if commit_hash:
            commit_id = (repo, commit_hash)
            if commit_id in seen_commits:
                duplicate_commits += 1
            seen_commits.add(commit_id)

        diff_lengths.append(len(diff_text))
        message_lengths.append(len(message))
        modification_counts.append(len(modifications))

        if not diff_text:
            empty_diffs += 1

        if not message:
            empty_messages += 1

        if message and message == original:
            same_as_original += 1

        for name, pattern in SENSITIVE_PATTERNS.items():
            if pattern.search(diff_text):
                sensitive_counts[name] += 1

        if len(previews) < 5:
            previews.append({
                "repository": repo,
                "language": language,
                "license": license_name,
                "message": message[:180],
                "modified_files": len(modifications),
                "diff_characters": len(diff_text),
            })

    return {
        "sampled_examples": len(records),
        "empty_diffs": empty_diffs,
        "empty_messages": empty_messages,
        "duplicate_commits": duplicate_commits,
        "message_equals_original": same_as_original,
        "diff_characters": distribution(diff_lengths),
        "message_characters": distribution(message_lengths),
        "modified_files_per_commit": distribution(modification_counts),
        "languages": dict(languages.most_common(20)),
        "licenses": dict(licenses.most_common(20)),
        "records_with_potential_sensitive_patterns": dict(
            sensitive_counts
        ),
        "examples_preview": previews,
    }


In [7]:
REPORT = {
    "dataset": DATASET_ID,
    "configuration": DATASET_CONFIG,
    "sample_size_requested": SAMPLE_SIZE,
    "seed": SEED,
    "splits": {},
}

for split_name, records in SAMPLES.items():
    print(f"\\n{'=' * 65}")
    print(f"ANALYSE : {split_name.upper()}")
    print("=" * 65)

    REPORT["splits"][split_name] = analyze_split(records)

    print(json.dumps(
        REPORT["splits"][split_name],
        indent=2,
        ensure_ascii=False,
    ))

output_path = Path(
    "data/interim/commit_chronicle_exploration.json"
)
output_path.parent.mkdir(parents=True, exist_ok=True)

output_path.write_text(
    json.dumps(REPORT, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\\nRapport enregistré :", output_path)


\n=================================================================
ANALYSE : TRAIN
{
  "sampled_examples": 1000,
  "empty_diffs": 0,
  "empty_messages": 0,
  "duplicate_commits": 0,
  "message_equals_original": 679,
  "diff_characters": {
    "count": 1000,
    "min": 159,
    "median": 1823,
    "p90": 9855,
    "p95": 15356,
    "max": 36458
  },
  "message_characters": {
    "count": 1000,
    "min": 11,
    "median": 44,
    "p90": 95,
    "p95": 137,
    "max": 359
  },
  "modified_files_per_commit": {
    "count": 1000,
    "min": 1,
    "median": 2,
    "p90": 6,
    "p95": 8,
    "max": 16
  },
  "languages": {
    "TypeScript": 289,
    "Python": 259,
    "Go": 132,
    "Java": 94,
    "C++": 89,
    "C#": 77,
    "JavaScript": 60
  },
  "licenses": {
    "Apache License 2.0": 515,
    "MIT License": 468,
    "BSD 3-Clause New or Revised License": 17
  },
  "records_with_potential_sensitive_patterns": {},
  "examples_preview": [
    {
      "repository": "layer5io/meshery",
 

## Résultat attendu

À la fin de cette exploration, nous disposerons de statistiques pour les
trois partitions.

Ces résultats serviront à déterminer les règles de nettoyage, les longueurs
maximales des entrées et des sorties, ainsi que le filtrage des licences.

**Important :** ces statistiques portent sur les échantillons examinés,
pas sur l'intégralité des 10,7 millions de commits.


In [8]:
from google.colab import files

files.download(
    "data/interim/commit_chronicle_exploration.json"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
%pip -q install transformers sentencepiece

In [10]:
from transformers import AutoTokenizer
import numpy as np

MODEL_NAME = "Salesforce/codet5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MODEL_LIMIT = 512

print("Modèle :", MODEL_NAME)
print("Limite du tokenizer :", tokenizer.model_max_length)


def serialize_commit(row):
    """
    Build a structured input containing the file path,
    change type and diff for every modified file.
    """
    sections = []

    for modification in row.get("mods") or []:
        old_path = modification.get("old_path")
        new_path = modification.get("new_path")
        change_type = modification.get("change_type", "UNKNOWN")
        diff = modification.get("diff") or ""

        if old_path and new_path and old_path != new_path:
            file_path = f"{old_path} -> {new_path}"
        else:
            file_path = new_path or old_path or "unknown"

        if not diff.strip():
            continue

        sections.append(
            f"### FILE: {file_path}\n"
            f"### CHANGE: {change_type}\n"
            f"{diff}"
        )

    return "\n\n".join(sections)


def token_length(text):
    if not text:
        return 0

    return len(
        tokenizer.encode(
            text,
            add_special_tokens=True,
            truncation=False,
        )
    )


def describe_lengths(values):
    values = np.asarray(values)

    return {
        "count": int(len(values)),
        "median": round(float(np.percentile(values, 50)), 2),
        "p90": round(float(np.percentile(values, 90)), 2),
        "p95": round(float(np.percentile(values, 95)), 2),
        "p99": round(float(np.percentile(values, 99)), 2),
        "maximum": int(values.max()),
    }


TOKEN_REPORT = {}

for split_name, records in SAMPLES.items():

    source_lengths = []
    processed_message_lengths = []
    original_message_lengths = []

    for row in records:
        source = serialize_commit(row)

        source_lengths.append(token_length(source))

        processed_message_lengths.append(
            token_length(str(row.get("message") or ""))
        )

        original_message_lengths.append(
            token_length(str(row.get("original_message") or ""))
        )

    source_lengths = np.asarray(source_lengths)

    TOKEN_REPORT[split_name] = {
        "source_tokens": describe_lengths(source_lengths),
        "processed_message_tokens": describe_lengths(
            processed_message_lengths
        ),
        "original_message_tokens": describe_lengths(
            original_message_lengths
        ),
        "source_over_512_tokens": int(
            np.sum(source_lengths > MODEL_LIMIT)
        ),
        "source_truncation_rate_percent": round(
            float(np.mean(source_lengths > MODEL_LIMIT) * 100),
            2,
        ),
        "processed_target_over_64_tokens": int(
            np.sum(np.asarray(processed_message_lengths) > 64)
        ),
        "original_target_over_64_tokens": int(
            np.sum(np.asarray(original_message_lengths) > 64)
        ),
    }

    print(f"\n{'=' * 60}")
    print(f"TOKEN ANALYSIS: {split_name.upper()}")
    print("=" * 60)

    for name, value in TOKEN_REPORT[split_name].items():
        print(f"{name}: {value}")

config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.48k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/703k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/294k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/12.5k [00:00<?, ?B/s]

TypeError: Input must be a List[Union[str, AddedToken]]

In [11]:
print("COMPARAISON DES MESSAGES TRAITÉS ET ORIGINAUX")
print("=" * 70)

differences = [
    row for row in SAMPLES["train"]
    if row.get("message") != row.get("original_message")
]

print("Nombre de différences :", len(differences))

for index, row in enumerate(differences[:10], start=1):

    print(f"\nEXEMPLE {index}")

    print("MESSAGE TRAITÉ :")
    print(str(row.get("message") or "")[:300])

    print("\nMESSAGE ORIGINAL :")
    print(str(row.get("original_message") or "")[:300])

    print("-" * 70)

COMPARAISON DES MESSAGES TRAITÉS ET ORIGINAUX
Nombre de différences : 321

EXEMPLE 1
MESSAGE TRAITÉ :
minor ui changes to the catalog

MESSAGE ORIGINAL :
minor ui changes to the catalog

Signed-off-by: Aditya Chatterjee <speak2adi@gmail.com>
----------------------------------------------------------------------

EXEMPLE 2
MESSAGE TRAITÉ :
Use idText in a few more places

MESSAGE ORIGINAL :
Use idText in a few more places (#18842)
----------------------------------------------------------------------

EXEMPLE 3
MESSAGE TRAITÉ :
Applying eslint checks and errors to the ui w/signoff

MESSAGE ORIGINAL :
Applying eslint checks and errors to the ui w/signoff

Signed-off-by: Kamal Nayan <95926324+legendarykamal@users.noreply.github.com>
----------------------------------------------------------------------

EXEMPLE 4
MESSAGE TRAITÉ :
docs: use classic function syntax
to avoid relying on implicit return,
which requires a higher level of JS understanding
as noted by

MESSAGE ORIGINAL :
docs: us

In [14]:
%pip -q install \
    "transformers==4.57.6" \
    "tokenizers>=0.22,<0.23" \
    "sentencepiece>=0.2,<0.3"

In [15]:
import transformers
import tokenizers

from transformers import AutoTokenizer

print("Transformers :", transformers.__version__)
print("Tokenizers :", tokenizers.__version__)

MODEL_NAME = "Salesforce/codet5-small"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

print("Classe :", type(tokenizer).__name__)
print("Limite du modèle :", tokenizer.model_max_length)

sample = "def calculate_total(price, quantity): return price * quantity"

encoded = tokenizer(
    sample,
    add_special_tokens=True,
    truncation=False,
)

print("Exemple de tokens :", encoded["input_ids"])
print("Nombre de tokens :", len(encoded["input_ids"]))

Transformers : 5.22.1
Tokenizers : 0.23.3


TypeError: Input must be a List[Union[str, AddedToken]]

In [1]:
%pip install --upgrade --no-cache-dir \
    "transformers==4.57.6" \
    "tokenizers==0.22.2" \
    "huggingface-hub>=0.34.0,<1.0" \
    "sentencepiece>=0.1.91,!=0.1.92"